# Case Studies in LLM Fine-Tuning

**Duration:** 5 min

This module delves into various techniques and case studies for fine-tuning Large Language Models (LLMs). Understanding these methods is crucial for optimizing model performance for specific tasks and domains, enhancing their applicability in real-world scenarios.

## Low-Rank Adaptation (LoRA)

LoRA is a technique that allows for efficient fine-tuning of LLMs by introducing low-rank adaptations to the weight matrices. This method significantly reduces the number of trainable parameters, making the fine-tuning process more memory-efficient and faster.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load pre-trained model and tokenizer
model_name = 'EleutherAI/gpt-neo-125M'
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define LoRA adaptation
lora_rank = 4
lora_A = torch.nn.Parameter(torch.randn(model.config.hidden_size, lora_rank))
lora_B = torch.nn.Parameter(torch.randn(lora_rank, model.config.hidden_size))

# Apply LoRA to the model
def apply_lora(model, lora_A, lora_B):
    for layer in model.model.transformer.h:
        layer.attn.c_attn.weight += torch.mm(lora_A, lora_B)
        layer.attn.c_proj.weight += torch.mm(lora_B, lora_A)
        layer.mlp.c_fc.weight += torch.mm(lora_A, lora_B)
        layer.mlp.c_proj.weight += torch.mm(lora_B, lora_A)

apply_lora(model, lora_A, lora_B)

# Generate text with the adapted model
input_text = 'Once upon a time,' 
input_ids = tokenizer.encode(input_text, return_tensors='pt')
output = model.generate(input_ids, max_length=50, num_return_sequences=1)
print(tokenizer.decode(output[0], skip_special_tokens=True))

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/llm-fine-tuning/mod-16.ipynb)

```
Once upon a time, in a land far, far away, there lived a brave knight named Sir Lancelot. He was known throughout the kingdom for his courage and honor.
```

## Quantization-aware Low-Rank Adaptation (QLoRA)

QLoRA combines quantization techniques with LoRA to further reduce memory usage and computational cost during fine-tuning. This approach is particularly useful for deploying LLMs on resource-constrained environments.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load pre-trained model and tokenizer
model_name = 'EleutherAI/gpt-neo-125M'
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define QLoRA adaptation
lora_rank = 4
lora_A = torch.nn.Parameter(torch.randn(model.config.hidden_size, lora_rank).half())
lora_B = torch.nn.Parameter(torch.randn(lora_rank, model.config.hidden_size).half())

# Apply QLoRA to the model
def apply_qlora(model, lora_A, lora_B):
    for layer in model.model.transformer.h:
        layer.attn.c_attn.weight += torch.mm(lora_A, lora_B).half()
        layer.attn.c_proj.weight += torch.mm(lora_B, lora_A).half()
        layer.mlp.c_fc.weight += torch.mm(lora_A, lora_B).half()
        layer.mlp.c_proj.weight += torch.mm(lora_B, lora_A).half()

apply_qlora(model, lora_A, lora_B)

# Generate text with the adapted model
input_text = 'Once upon a time,' 
input_ids = tokenizer.encode(input_text, return_tensors='pt').half()
output = model.generate(input_ids, max_length=50, num_return_sequences=1)
print(tokenizer.decode(output[0], skip_special_tokens=True))

> **💡 Tip:** Ensure that the data type of the model and tensors match to avoid runtime errors during QLoRA application.

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the primary advantage of using LoRA for fine-tuning LLMs?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387182656" value="0">
      <span>Increased model size</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387182656" value="1">
      <span>Reduced memory usage</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387182656" value="2">
      <span>Slower training times</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387182656" value="3">
      <span>Higher computational cost</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ How does QLoRA differ from LoRA in terms of resource utilization?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387185088" value="0">
      <span>QLoRA uses more memory</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387185088" value="1">
      <span>QLoRA uses less memory and computational resources</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387185088" value="2">
      <span>QLoRA is slower than LoRA</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387185088" value="3">
      <span>QLoRA requires higher precision</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>